# NL → Cypher Pipeline für SAGA.ch Wissensgraph

**Aufbau:**
1. Verbindung zu Neo4j und Anthropic Claude API
2. Schema-Beschreibung und Few-Shot-Beispiele
3. Pipeline-Funktion `generate_cypher()`
4. Testset von 12 deutschen Fragen × 4 Anwendungsfall-Klassen × 3 Komplexitätsstufen
5. Evaluation: **Korrektheit** (binär) und **Ergebnisqualität** (3-Punkt-Skala)
6. Stabilitätstest (3 Wiederholungen je Frage, vgl. §4.3.1)
7. Aggregierte Metriken

## 1 · Setup und Verbindungen

Erforderliche Umgebungsvariablen:

```bash
export NEO4J_URI=bolt://localhost:7687
export NEO4J_USER=neo4j
export NEO4J_PASSWORD=<dein_passwort>
export ANTHROPIC_API_KEY=<dein_anthropic_key>
```

In [2]:
import os
import json
import time
from dataclasses import dataclass

import pandas as pd
from anthropic import Anthropic
from neo4j import GraphDatabase

# --- Neo4j ---
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]
neo4j_driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
neo4j_driver.verify_connectivity()
print(f"✓ Neo4j verbunden: {NEO4J_URI}")

# --- Anthropic ---
anthropic_client = Anthropic()
# 'claude-opus-4-7' lehnt das temperature-Argument ab und macht den
# Stabilitätstest in §9 unmöglich. Sonnet 4.6 unterstützt temperature und
# ist mehr als ausreichend für deutsche NL→Cypher-Übersetzung.
MODEL = "claude-sonnet-4-6"
print(f"✓ Anthropic Client bereit, Modell: {MODEL}")

KeyError: 'NEO4J_PASSWORD'

## 2 · Graphschema für den LLM-Prompt

Die Schema-Beschreibung wird in den System-Prompt eingebettet. Sie ist auf 
Deutsch verfasst, weil die Daten und das Testset deutschsprachig sind.

In [2]:
SCHEMA_DE = """
GRAPHSCHEMA - SAGA.ch (eCH-0014) Wissensgraph in Neo4j 5
========================================================

KNOTEN (mit Eigenschaften):

  (:Document {id, title, version})
      Das SAGA.ch-Dokument.

  (:Building_Block:Macro {id, title, semantic_summary})
      Top-Level fachlicher Bereich. 4 Knoten: Kommunikationsprotokolle,
      Datei- und Datenbeschreibungsformate, Sicherheit, Querschnittsthemen.

  (:Building_Block:Meso {id, title, semantic_summary})
      Thematische Untergruppe innerhalb eines Macros (32 Knoten).

  (:Building_Block:Mikro {id, title, semantic_summary})
      Einzelne Standards/Technologien wie HTTP, SOAP, LDAP (186 Knoten).
      Wichtige IDs zum Beispiel: bb_mi_http, bb_mi_soap, bb_mi_ldap, bb_mi_rest,
      bb_mi_smtp, bb_mi_ftp, bb_mi_telnet, bb_mi_corba, bb_mi_sedex.

  (:Variant {id, title})
      Versionen eines Mikro-Bausteins (z.B. HTTP V1.1 vs HTTP V2). 113 Knoten.

  (:Alternative {id, title})
      Sich gegenseitig ausschliessende Optionen (z.B. POP3 ODER IMAP4).
      14 Knoten.

  (:External_Standard {key, label, organization, url})
      Externe Standards (z.B. key='IETF RFC 7540', organization='IETF').

  (:Interface {id, description})
      3 Knoten mit id ∈ {"S1", "S2", "S3"}.

BEZIEHUNGEN:

  (:Document)-[:DEFINES]->(:Macro)
  (:Macro)-[:CONTAINS]->(:Meso)
  (:Meso)-[:CONTAINS]->(:Mikro)
  (:Mikro)-[:HAS_VARIANT]->(:Variant)
  (:Mikro)-[:HAS_ALTERNATIVE]->(:Alternative)
  (:Variant|:Alternative|:Mikro)-[:REFERENCES {reference_type}]->(:External_Standard)
  (:Variant|:Alternative|:Mikro)-[:APPLIES_TO {normative_status}]->(:Interface)

WERTEBEREICHE:

  normative_status ∈ {"dringend empfohlen", "empfohlen", "nicht empfohlen", "unter beobachtung"}
  Interface.id     ∈ {"S1", "S2", "S3"}
  reference_type   ∈ {"standard", "spezifikation", "empfehlung", ...}

WICHTIGE HINWEISE:

  - Bewertungen (APPLIES_TO) hängen ÜBLICHERWEISE an Variant oder Alternative,
    NICHT direkt am Mikro. Bei Abfragen über Bewertungen eines Mikros muss
    über HAS_VARIANT/HAS_ALTERNATIVE traversiert werden.
  - Macro/Meso/Mikro tragen alle das Label :Building_Block plus ihr Level-Label.
  - Macro hat KEINE Bewertungen oder Referenzen direkt - nur Mikros, Varianten
    und Alternativen tragen diese.
"""
print(f"Schema-Länge: {len(SCHEMA_DE)} Zeichen")

Schema-Länge: 2261 Zeichen


## 3 · Few-Shot-Beispiele

Drei Beispielpaare in Neo4j-5-konformer Syntax. Wichtig: bei alternativen 
Beziehungstypen wird `[:HAS_VARIANT|HAS_ALTERNATIVE]` verwendet (ohne zweiten 
Doppelpunkt), da Neo4j 5 die Form `[:R1|:R2]` mit variabler Länge nicht mehr 
akzeptiert.

In [3]:
FEW_SHOT_EXAMPLES = [
    {
        "question": "Welche Varianten hat der Mikro-Block HTTP?",
        "cypher": (
            "MATCH (m:Mikro {id:'bb_mi_http'})-[:HAS_VARIANT]->(v:Variant)\n"
            "RETURN v.id AS variant_id, v.title AS variant_title\n"
            "ORDER BY v.title"
        ),
    },
    {
        "question": "Was ist für die Schnittstelle S1 dringend empfohlen?",
        "cypher": (
            "MATCH (s)-[:APPLIES_TO {normative_status:'dringend empfohlen'}]\n"
            "      ->(:Interface {id:'S1'})\n"
            "OPTIONAL MATCH (s)<-[:HAS_VARIANT|HAS_ALTERNATIVE]-(m:Mikro)\n"
            "RETURN coalesce(m.id, s.id) AS id, coalesce(m.title, s.title) AS title,\n"
            "       labels(s) AS source_kind\n"
            "ORDER BY title"
        ),
    },
    {
        "question": "Welche externen Standards werden vom Mikro-Block SOAP referenziert?",
        "cypher": (
            "MATCH (m:Mikro {id:'bb_mi_soap'})-[:HAS_VARIANT]->(v:Variant)\n"
            "      -[r:REFERENCES]->(e:External_Standard)\n"
            "RETURN DISTINCT e.key AS standard, e.organization AS org\n"
            "ORDER BY standard"
        ),
    },
]
print(f"{len(FEW_SHOT_EXAMPLES)} Few-Shot-Beispiele definiert.")

3 Few-Shot-Beispiele definiert.


## 4 · Pipeline-Funktion `generate_cypher()`

Der System-Prompt enthält:
- Das Schema
- Die Few-Shot-Beispiele
- Strikte Format-Regeln (kein Markdown, keine Erklärung)
- **Hinweis zur Neo4j-5-Syntax** (`[:R1|R2]`, nicht `[:R1|:R2]`)

`temperature=0` für maximalen Determinismus (Voraussetzung für den 
Stabilitätstest in §9).

In [4]:
SYSTEM_PROMPT_TEMPLATE = """Du bist ein Experte für Neo4j Cypher-Abfragen
(Neo4j 5). Generiere syntaktisch korrekte und ausführbare Cypher-Abfragen für
den folgenden Wissensgraphen.

{schema}

REGELN:
1. Antworte AUSSCHLIESSLICH mit der Cypher-Abfrage. Keine Erklärung, kein
   Markdown-Code-Fence (kein ``` davor oder dahinter), kein Kommentar.
2. Nutze nur die im Schema definierten Labels und Beziehungen.
3. normative_status-Werte und Interface-IDs müssen exakt wie im Schema
   geschrieben werden (Kleinschreibung bei normative_status).
4. Bei eindeutigen Mikro-Bausteinen wie "HTTP", "SOAP", "LDAP", "REST"
   bevorzuge die exakte ID-Übereinstimmung (z.B. m.id = 'bb_mi_http').
   toLower() + CONTAINS nur als letzte Option, wenn die ID nicht
   eindeutig ableitbar ist.
5. Bei alternativen Beziehungstypen verwende die Neo4j-5-Syntax
   [:HAS_VARIANT|HAS_ALTERNATIVE] (OHNE zweiten Doppelpunkt). Die alte
   Form [:HAS_VARIANT|:HAS_ALTERNATIVE] wird mit variabler Länge nicht mehr
   akzeptiert.
6. Bei Resultaten: gib bei Entitäten möglichst BEIDE id UND title zurück,
   damit das Resultat unabhängig von der Repräsentation lesbar bleibt.

BEISPIELE:

{examples}
"""

def build_system_prompt() -> str:
    examples_str = "\n\n".join(
        f"Frage: {ex['question']}\nCypher:\n{ex['cypher']}"
        for ex in FEW_SHOT_EXAMPLES
    )
    return SYSTEM_PROMPT_TEMPLATE.format(schema=SCHEMA_DE, examples=examples_str)


def extract_cypher(text: str) -> str:
    """Markdown-Code-Fence entfernen, falls vorhanden."""
    text = text.strip()
    if text.startswith("```"):
        lines = text.split("\n")
        if lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        text = "\n".join(lines)
    return text.strip()


def generate_cypher(question: str, temperature: float = 0.0) -> str:
    response = anthropic_client.messages.create(
        model=MODEL,
        max_tokens=1024,
        temperature=temperature,
        system=build_system_prompt(),
        messages=[{"role": "user", "content": question}],
    )
    return extract_cypher(response.content[0].text)

## 5 · End-to-End-Demonstration

Schnelltest mit einer einfachen Frage. Zeigt den vollständigen Pfad: 
NL-Frage → generierter Cypher → Neo4j-Resultat.

In [5]:
def execute_cypher(query: str) -> tuple[list[dict], str | None]:
    try:
        with neo4j_driver.session() as session:
            return [dict(r) for r in session.run(query)], None
    except Exception as e:
        return [], f"{type(e).__name__}: {e}"


demo_question = "Welche Varianten hat der Mikro-Block HTTP?"
print(f"Frage: {demo_question}\n")

generated = generate_cypher(demo_question)
print(f"Generierter Cypher:\n{generated}\n")

records, error = execute_cypher(generated)
if error:
    print(f"FEHLER: {error}")
else:
    print(f"Resultat ({len(records)} Zeile(n)):")
    for r in records:
        print(f"  {r}")

Frage: Welche Varianten hat der Mikro-Block HTTP?

Generierter Cypher:
MATCH (m:Mikro {id:'bb_mi_http'})-[:HAS_VARIANT]->(v:Variant)
RETURN v.id AS variant_id, v.title AS variant_title
ORDER BY v.title

Resultat (2 Zeile(n)):
  {'variant_id': 'var_http_v1_1', 'variant_title': 'HTTP V1.1'}
  {'variant_id': 'var_http_v2', 'variant_title': 'HTTP V2'}


## 6 · Testset

12 deutsche Fragen × **4 Anwendungsfall-Klassen** × **3 Komplexitätsstufen** 
(`low` / `mid` / `high`).

Jeder Fall führt eine Referenz-Cypher (Ground Truth) mit. Diese Referenz-Cypher 
gibt für Entitäten **sowohl `id` als auch `title`** zurück (sofern sinnvoll), 
damit der Token-Overlap-Vergleich auch dann funktioniert, wenn das 
LLM eine alternative Repräsentation wählt.

| Klasse | Beschreibung |
|--------|--------------|
| **A** | Traceability — Standardlokalisierung und Querverweise |
| **B** | Dependency / Impact-Analyse — Rückverweise und Abhängigkeiten |
| **C** | Normative Status — Bewertung pro Schnittstelle |
| **D** | Coverage / Daten-Hygiene |

In [6]:
TEST_CASES = [
    # === A · Traceability ===
    {
        "id": "A1", "category": "Traceability", "complexity": "low",
        "question": "An welcher Stelle im Standard befindet sich der Mikro-Block HTTP (Hyper Text Transfer Protocol)? Zeige den Pfad vom Document über Macro und Meso bis zum Mikro.",
        "ground_truth": (
            "MATCH (d:Document)-[:DEFINES]->(macro:Macro)-[:CONTAINS]->(meso:Meso)\n"
            "      -[:CONTAINS]->(m:Mikro {id:'bb_mi_http'})\n"
            "RETURN d.title AS document, macro.title AS macro,\n"
            "       meso.title AS meso, m.title AS mikro_title, m.id AS mikro_id"
        ),
    },
    {
        "id": "A2", "category": "Traceability", "complexity": "mid",
        "question": "Welche externen Standards werden vom Mikro-Block SOAP über alle Varianten referenziert? Gib die Schlüssel der Standards zurück.",
        "ground_truth": (
            "MATCH (m:Mikro {id:'bb_mi_soap'})-[:HAS_VARIANT]->(v:Variant)\n"
            "      -[:REFERENCES]->(e:External_Standard)\n"
            "RETURN DISTINCT e.key AS standard ORDER BY e.key"
        ),
    },
    {
        "id": "A3", "category": "Traceability", "complexity": "high",
        "question": "Liste alle Mikro-Bausteine im Macro-Bereich Sicherheit auf, die mindestens eine Variante mit dem Status 'dringend empfohlen' besitzen. Gib id und title zurück.",
        "ground_truth": (
            "MATCH (:Macro {id:'bb_ma_sicherheit'})-[:CONTAINS]->(:Meso)\n"
            "      -[:CONTAINS]->(m:Mikro)-[:HAS_VARIANT]->(v:Variant)\n"
            "      -[:APPLIES_TO {normative_status:'dringend empfohlen'}]->(:Interface)\n"
            "RETURN DISTINCT m.id AS mikro_id, m.title AS mikro_title\n"
            "ORDER BY m.id"
        ),
    },
    # === B · Dependency / Impact ===
    {
        "id": "B1", "category": "Dependency", "complexity": "low",
        "question": "Welche Building Blocks (Mikros, Variants, Alternatives) referenzieren den externen Standard mit Schlüssel 'IETF RFC 7540'? Gib id und title zurück.",
        "ground_truth": (
            "MATCH (e:External_Standard {key:'IETF RFC 7540'})<-[:REFERENCES]-(s)\n"
            "OPTIONAL MATCH (s)<-[:HAS_VARIANT|HAS_ALTERNATIVE]-(m:Mikro)\n"
            "RETURN coalesce(m.id, s.id) AS bb_id,\n"
            "       coalesce(m.title, s.title) AS bb_title"
        ),
    },
    {
        "id": "B2", "category": "Dependency", "complexity": "mid",
        "question": "Welche externen Standards werden von mehr als einem Mikro-Block referenziert? Sortiere absteigend nach Anzahl, gib Top 10 zurück.",
        "ground_truth": (
            "MATCH (e:External_Standard)<-[:REFERENCES]-(s)\n"
            "OPTIONAL MATCH (s)<-[:HAS_VARIANT|HAS_ALTERNATIVE]-(m:Mikro)\n"
            "WITH e, count(DISTINCT coalesce(m.id, s.id)) AS users\n"
            "WHERE users > 1\n"
            "RETURN e.key AS standard, users\n"
            "ORDER BY users DESC, standard LIMIT 10"
        ),
    },
    {
        "id": "B3", "category": "Dependency", "complexity": "high",
        "question": "Wie viele unterschiedliche Mikro-Bausteine wären betroffen, wenn alle Standards der Organisation IETF überarbeitet würden?",
        "ground_truth": (
            "MATCH (e:External_Standard {organization:'IETF'})<-[:REFERENCES]-(s)\n"
            "MATCH (s)<-[:HAS_VARIANT|HAS_ALTERNATIVE*0..1]-(m:Mikro)\n"
            "RETURN count(DISTINCT m) AS affected_mikros"
        ),
    },
    # === C · Normative Status ===
    {
        "id": "C1", "category": "Normative", "complexity": "low",
        "question": "Welche Variants und Alternatives sind für die Schnittstelle S1 dringend empfohlen? Gib id, title und labels zurück.",
        "ground_truth": (
            "MATCH (s)-[:APPLIES_TO {normative_status:'dringend empfohlen'}]\n"
            "      ->(:Interface {id:'S1'})\n"
            "RETURN s.id AS source_id, s.title AS source_title,\n"
            "       labels(s) AS kind\n"
            "ORDER BY source_title"
        ),
    },
    {
        "id": "C2", "category": "Normative", "complexity": "mid",
        "question": "Welche Mikro-Bausteine sind insgesamt nicht empfohlen (auf irgendeiner Schnittstelle)? Gib id und title zurück.",
        "ground_truth": (
            "MATCH (s)-[:APPLIES_TO {normative_status:'nicht empfohlen'}]->(:Interface)\n"
            "OPTIONAL MATCH (s)<-[:HAS_VARIANT|HAS_ALTERNATIVE]-(m:Mikro)\n"
            "RETURN DISTINCT coalesce(m.id, s.id) AS source_id,\n"
            "                coalesce(m.title, s.title) AS source_title\n"
            "ORDER BY source_id"
        ),
    },
    {
        "id": "C3", "category": "Normative", "complexity": "high",
        "question": "Wie viele Varianten gibt es pro normativem Status, gruppiert nach Macro-Bereich?",
        "ground_truth": (
            "MATCH (ma:Macro)-[:CONTAINS]->(:Meso)-[:CONTAINS]->(:Mikro)\n"
            "      -[:HAS_VARIANT]->(v:Variant)-[a:APPLIES_TO]->()\n"
            "RETURN ma.title AS macro, a.normative_status AS status,\n"
            "       count(DISTINCT v) AS variant_count\n"
            "ORDER BY macro, status"
        ),
    },
    # === D · Coverage / Hygiene ===
    {
        "id": "D1", "category": "Coverage", "complexity": "low",
        "question": "Welche Mikro-Bausteine haben überhaupt keine Bewertung (weder direkt am Mikro noch über Variant oder Alternative)? Gib id und title zurück.",
        "ground_truth": (
            "MATCH (m:Mikro)\n"
            "WHERE NOT EXISTS { (m)-[:APPLIES_TO]->() }\n"
            "  AND NOT EXISTS { (m)-[:HAS_VARIANT|HAS_ALTERNATIVE]->()-[:APPLIES_TO]->() }\n"
            "RETURN m.id AS mikro_id, m.title AS mikro_title\n"
            "ORDER BY m.id"
        ),
    },
    {
        "id": "D2", "category": "Coverage", "complexity": "mid",
        "question": "Welche externen Standards werden nur ein einziges Mal referenziert?",
        "ground_truth": (
            "MATCH (e:External_Standard)<-[:REFERENCES]-()\n"
            "WITH e, count(*) AS users\n"
            "WHERE users = 1\n"
            "RETURN e.key AS standard, e.organization AS organization\n"
            "ORDER BY standard"
        ),
    },
    {
        "id": "D3", "category": "Coverage", "complexity": "high",
        "question": "Wie viele Mikro-Bausteine pro Macro-Bereich gibt es, und wie viele davon haben mindestens eine Variante?",
        "ground_truth": (
            "MATCH (ma:Macro)-[:CONTAINS]->(:Meso)-[:CONTAINS]->(m:Mikro)\n"
            "WITH ma, count(DISTINCT m) AS mikro_total,\n"
            "     count(DISTINCT CASE WHEN EXISTS { (m)-[:HAS_VARIANT]->() }\n"
            "                         THEN m END) AS mit_variante\n"
            "RETURN ma.title AS macro, mikro_total, mit_variante\n"
            "ORDER BY macro"
        ),
    },
]
print(f"{len(TEST_CASES)} Testfälle definiert.")

12 Testfälle definiert.


## 7 · Evaluations-Hilfsfunktionen

**Per-Row Token-Overlap-Scoring** statt strikter Set-Vergleich. Dieses Verfahren 
erkennt semantisch korrekte Antworten auch dann, wenn das LLM:

- zusätzliche Informationsspalten liefert (z.B. `mikro_title` neben `mikro_id`),
- andere Spaltennamen verwendet (`bb_id` vs `id`),
- eine alternative Repräsentation der Entität wählt (z.B. nur den Title statt der ID).

**Score-Skala (Disposition §4.3.1, "Ergebnisqualität"):**

| Score | Bedeutung | Kriterium |
|-------|-----------|-----------|
| **2** | vollständig korrekt | Alle erwarteten Zeilen gefunden, Zeilenanzahl ≤ 1.5× erwartet |
| **1** | teilweise korrekt | Alle gefunden aber zu viele Zeilen, ODER ≥ 50% gefunden |
| **0** | nicht korrekt | < 50% gefunden, leeres Resultat oder Cypher-Fehler |

Korrektheit = `score >= 1` UND keine Ausführungsfehler.

**"Signifikanter Token"** = Token mit Länge ≥ 3 Zeichen ODER reine Zahl. Damit 
werden Bindewörter wie "the", "und", "von" beim Vergleich ignoriert, aber 
Mikro-IDs (`bb_mi_http`), Standard-Schlüssel (`RFC 7540`), Status-Werte 
(`empfohlen`) und numerische Counts werden berücksichtigt.

In [7]:
def _flatten_tokens(value) -> set[str]:
    """Wert -> Set von kleingeschriebenen String-Tokens."""
    if value is None:
        return set()
    if isinstance(value, list):
        return {str(x).lower().strip() for x in value if x is not None}
    return {str(value).lower().strip()}


def _row_tokens(record: dict) -> set[str]:
    tokens = set()
    for v in record.values():
        tokens |= _flatten_tokens(v)
    return tokens


def _significant(tokens: set[str], min_len: int = 3) -> set[str]:
    """Tokens mit Länge >= 3 ODER reine Zahl (für Counts wie '47')."""
    return {t for t in tokens if len(t) >= min_len or t.isdigit()}


def score_results(actual: list[dict], expected: list[dict]) -> int:
    """0/1/2 Bewertung gemäss Disposition §4.3.1 (per-row token overlap)."""
    if not expected:
        return 2 if not actual else 0
    if not actual:
        return 0

    actual_sets = [_significant(_row_tokens(r)) for r in actual]
    expected_sets = [_significant(_row_tokens(r)) for r in expected]
    expected_non_empty = [s for s in expected_sets if s]

    if not expected_non_empty:
        return 2 if not actual else 0

    matched = sum(
        1 for exp in expected_non_empty
        if any(exp & act for act in actual_sets)
    )
    coverage = matched / len(expected_non_empty)

    if coverage == 1.0 and len(actual) <= len(expected) * 1.5:
        return 2
    if coverage == 1.0 or coverage >= 0.5:
        return 1
    return 0


@dataclass
class EvalResult:
    test_id: str
    category: str
    complexity: str
    question: str
    generated_cypher: str
    error: str | None
    actual_rows: int
    expected_rows: int
    score: int
    correct: bool
    duration_s: float


def run_test_case(case: dict, temperature: float = 0.0) -> EvalResult:
    t0 = time.time()
    generated = generate_cypher(case["question"], temperature=temperature)
    actual, err = execute_cypher(generated)
    expected, gt_err = execute_cypher(case["ground_truth"])
    if gt_err:
        raise RuntimeError(f"Ground-Truth fehlerhaft für {case['id']}: {gt_err}")
    score = 0 if err else score_results(actual, expected)
    return EvalResult(
        test_id=case["id"], category=case["category"], complexity=case["complexity"],
        question=case["question"], generated_cypher=generated, error=err,
        actual_rows=len(actual), expected_rows=len(expected),
        score=score, correct=(err is None and score >= 1),
        duration_s=round(time.time() - t0, 2),
    )

## 8 · Hauptevaluation (12 Testfälle, 1 Lauf)

Alle Testfälle werden einmal durchlaufen.

In [8]:
results = []
for case in TEST_CASES:
    print(f"  Laufe {case['id']} ({case['category']}, {case['complexity']}) ...", end=" ")
    try:
        r = run_test_case(case)
        print(f"score={r.score}, correct={r.correct}, {r.duration_s}s"
              + (f", FEHLER: {r.error}" if r.error else ""))
        results.append(r)
    except Exception as e:
        print(f"ABGEBROCHEN: {e}")
print(f"\n{len(results)} Testfälle erfolgreich ausgeführt.")

  Laufe A1 (Traceability, low) ... score=2, correct=True, 2.96s
  Laufe A2 (Traceability, mid) ... score=2, correct=True, 2.61s
  Laufe A3 (Traceability, high) ... score=2, correct=True, 3.02s
  Laufe B1 (Dependency, low) ... score=0, correct=False, 2.17s
  Laufe B2 (Dependency, mid) ... score=0, correct=False, 4.86s
  Laufe B3 (Dependency, high) ... score=2, correct=True, 2.89s
  Laufe C1 (Normative, low) ... score=1, correct=True, 3.0s
  Laufe C2 (Normative, mid) ... score=2, correct=True, 2.72s
  Laufe C3 (Normative, high) ... score=2, correct=True, 2.82s
  Laufe D1 (Coverage, low) ... score=2, correct=True, 2.92s
  Laufe D2 (Coverage, mid) ... score=2, correct=True, 3.14s
  Laufe D3 (Coverage, high) ... score=2, correct=True, 3.6s

12 Testfälle erfolgreich ausgeführt.


### Resultatstabelle

In [9]:
df = pd.DataFrame([{
    "Test": r.test_id, "Kategorie": r.category, "Komplexität": r.complexity,
    "Score": r.score, "Korrekt": "✓" if r.correct else "✗",
    "Zeilen erwartet": r.expected_rows, "Zeilen tatsächlich": r.actual_rows,
    "Dauer [s]": r.duration_s,
    "Fehler": (r.error[:60] + "...") if r.error and len(r.error) > 60 else (r.error or ""),
} for r in results])
df

,Test,Kategorie,Komplexität,Score,Korrekt,Zeilen erwartet,Zeilen tatsächlich,Dauer [s],Fehler
0,A1,Traceability,low,2,✓,1,1,2.96,
1,A2,Traceability,mid,2,✓,3,3,2.61,
2,A3,Traceability,high,2,✓,4,4,3.02,
3,B1,Dependency,low,0,✗,1,1,2.17,
4,B2,Dependency,mid,0,✗,7,10,4.86,
5,B3,Dependency,high,2,✓,1,1,2.89,
6,C1,Normative,low,1,✓,58,33,3.00,
7,C2,Normative,mid,2,✓,21,21,2.72,
8,C3,Normative,high,2,✓,11,11,2.82,
9,D1,Coverage,low,2,✓,2,2,2.92,


### Detaileinsicht in einen einzelnen Testfall

In [10]:
target_id = "A1"
match = [r for r in results if r.test_id == target_id]
if match:
    r = match[0]
    print("=" * 70)
    print(f"Testfall: {r.test_id} ({r.category} / {r.complexity})")
    print(f"Score: {r.score} | Korrekt: {r.correct}")
    print("=" * 70)
    print(f"\nFRAGE:\n  {r.question}\n")
    print(f"GENERIERTER CYPHER:\n{r.generated_cypher}\n")
    if r.error:
        print(f"FEHLER: {r.error}\n")
    else:
        actual, _ = execute_cypher(r.generated_cypher)
        print(f"GENERIERTES RESULTAT ({len(actual)} Zeilen):")
        for row in actual[:10]:
            print(f"  {row}")
        if len(actual) > 10:
            print(f"  ... ({len(actual) - 10} weitere)")

        expected_q = next(c for c in TEST_CASES if c["id"] == target_id)["ground_truth"]
        expected, _ = execute_cypher(expected_q)
        print(f"\nGROUND-TRUTH-RESULTAT ({len(expected)} Zeilen):")
        for row in expected[:10]:
            print(f"  {row}")
else:
    print(f"Kein Resultat für {target_id} gefunden.")

Testfall: A1 (Traceability / low)
Score: 2 | Korrekt: True

FRAGE:
  An welcher Stelle im Standard befindet sich der Mikro-Block HTTP (Hyper Text Transfer Protocol)? Zeige den Pfad vom Document über Macro und Meso bis zum Mikro.

GENERIERTER CYPHER:
MATCH (doc:Document)-[:DEFINES]->(mac:Building_Block:Macro)-[:CONTAINS]->(mes:Building_Block:Meso)-[:CONTAINS]->(mi:Building_Block:Mikro {id: 'bb_mi_http'})
RETURN doc.id AS document_id, doc.title AS document_title,
       mac.id AS macro_id, mac.title AS macro_title,
       mes.id AS meso_id, mes.title AS meso_title,
       mi.id AS mikro_id, mi.title AS mikro_title

GENERIERTES RESULTAT (1 Zeilen):
  {'document_id': 'eCH-0014', 'document_title': 'SAGA.ch', 'macro_id': 'bb_ma_kommunikationsprotokolle', 'macro_title': 'Kommunikationsprotokolle', 'meso_id': 'bb_me_anwendungsprotokolle', 'meso_title': 'Anwendungsprotokolle', 'mikro_id': 'bb_mi_http', 'mikro_title': 'Hyper Text Transfer Protocol, HTTP'}

GROUND-TRUTH-RESULTAT (1 Zeilen):
  {'d

## 9 · Stabilitätstest

Drei Testfälle (`low` / `mid` / `high`) werden je dreimal ausgeführt.

Bei `temperature=0` erwarten wir hohe Konsistenz; Abweichungen zeigen 
nicht-deterministische Streuung im Sampling und werden in der Diskussion 
erläutert.

In [11]:
STABILITY_IDS = ["A1", "B2", "C3"]
N_RUNS = 3

stability_records = []
for tid in STABILITY_IDS:
    case = next(c for c in TEST_CASES if c["id"] == tid)
    cyphers, scores = [], []
    print(f"  Stabilität {tid}: ", end="")
    for run in range(N_RUNS):
        r = run_test_case(case, temperature=0.0)
        cyphers.append(r.generated_cypher)
        scores.append(r.score)
        print(f"[{run+1}: score={r.score}] ", end="")
    print()
    stability_records.append({
        "Test": tid,
        "Komplexität": case["complexity"],
        "Cypher konstant": "✓" if len(set(cyphers)) == 1 else "✗",
        "Scores": scores,
        "Score-Konsistenz": "✓" if len(set(scores)) == 1 else "✗",
    })
pd.DataFrame(stability_records)

  Stabilität A1: [1: score=2] [2: score=2] [3: score=2] 
  Stabilität B2: [1: score=0] [2: score=0] [3: score=0] 
  Stabilität C3: [1: score=2] [2: score=2] [3: score=2] 


,Test,Komplexität,Cypher konstant,Scores,Score-Konsistenz
0,A1,low,✓,"[2, 2, 2]",✓
1,B2,mid,✓,"[0, 0, 0]",✓
2,C3,high,✗,"[2, 2, 2]",✓


## 10 · Aggregierte Metriken (Disposition §4.3.1)

Drei Metriken aus dem Evaluationsplan:

1. **Korrektheit der Abfrageunterstützung** (binär): `correct == True`
2. **Ergebnisqualität** (3-Punkt-Skala): durchschnittlicher Score
3. **Stabilität**: Konsistenz der Wiederholungen oben

In [12]:
total = len(results)
n_correct = sum(1 for r in results if r.correct)
mean_score = sum(r.score for r in results) / total if total else 0

print("=" * 60)
print("AGGREGIERTE METRIKEN")
print("=" * 60)
print(f"Korrektheit (binär)   : {n_correct}/{total} = {n_correct/max(total,1):.1%}")
print(f"Ergebnisqualität (Ø)  : {mean_score:.2f} / 2.00")
print()
print("Pro Kategorie:")
for cat in ["Traceability", "Dependency", "Normative", "Coverage"]:
    cr = [r for r in results if r.category == cat]
    if cr:
        c = sum(1 for r in cr if r.correct)
        m = sum(r.score for r in cr) / len(cr)
        print(f"  {cat:14s}: korrekt {c}/{len(cr)}, Ø-Score {m:.2f}")
print()
print("Pro Komplexität:")
for cx in ["low", "mid", "high"]:
    cr = [r for r in results if r.complexity == cx]
    if cr:
        c = sum(1 for r in cr if r.correct)
        m = sum(r.score for r in cr) / len(cr)
        print(f"  {cx:14s}: korrekt {c}/{len(cr)}, Ø-Score {m:.2f}")

AGGREGIERTE METRIKEN
Korrektheit (binär)   : 10/12 = 83.3%
Ergebnisqualität (Ø)  : 1.58 / 2.00

Pro Kategorie:
  Traceability  : korrekt 3/3, Ø-Score 2.00
  Dependency    : korrekt 1/3, Ø-Score 0.67
  Normative     : korrekt 3/3, Ø-Score 1.67
  Coverage      : korrekt 3/3, Ø-Score 2.00

Pro Komplexität:
  low           : korrekt 3/4, Ø-Score 1.25
  mid           : korrekt 3/4, Ø-Score 1.50
  high          : korrekt 4/4, Ø-Score 2.00


## 11 · Ergebnisse persistieren

In [13]:
out = {
    "model": MODEL,
    "n_cases": len(results),
    "korrektheit": n_correct / max(total, 1),
    "mean_score": mean_score,
    "stability": stability_records,
    "cases": [
        {
            "id": r.test_id, "category": r.category, "complexity": r.complexity,
            "question": r.question, "generated_cypher": r.generated_cypher,
            "score": r.score, "correct": r.correct,
            "actual_rows": r.actual_rows, "expected_rows": r.expected_rows,
            "error": r.error, "duration_s": r.duration_s,
        } for r in results
    ],
}
with open("evaluation_results.json", "w", encoding="utf-8") as f:
    json.dump(out, f, indent=2, ensure_ascii=False)
print("✓ evaluation_results.json gespeichert")

✓ evaluation_results.json gespeichert


## 12 · Aufräumen

In [14]:
neo4j_driver.close()
print("✓ Neo4j-Verbindung geschlossen.")

✓ Neo4j-Verbindung geschlossen.
